# 1. Producing the data  
In this part, I implemented Apache Kafka producers to simulate real-time data streaming.

In [ ]:
# Import statements
from time import sleep
from json import dumps
from kafka import KafkaProducer
import random
import csv
import datetime as dt


# Kafka configuration
hostip = "kafka"
topic = "a2b_accident_stream"


def read_csv_file(file_name):
    """
    Read all rows from the collision CSV file as dictionaries.
    All values are kept as strings here, because type conversion will be handled
    later in Spark Structured Streaming.
    """
    rows = []

    with open(file_name, "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)

        for row in reader:
            rows.append(row)

    return rows


def connect_kafka_producer():
    """
    Create and return a Kafka producer.
    The message value is serialised into JSON format.
    """
    producer = None

    try:
        producer = KafkaProducer(
            bootstrap_servers=[f"{hostip}:9092"],
            value_serializer=lambda x: dumps(x).encode("ascii"),
            api_version=(3, 9)
        )

        print("Kafka producer connected successfully.")

    except Exception as ex:
        print("Exception while connecting Kafka.")
        print(str(ex))

    finally:
        return producer


def publish_message(producer_instance, topic_name, data):
    """
    Publish one batch of accident records to the Kafka topic.
    """
    try:
        producer_instance.send(topic_name, data)
        producer_instance.flush()

        print(
            f"Published batch successfully. "
            f"Records: {len(data)}, "
            f"accident_ts: {data[0]['accident_ts']}"
        )

    except Exception as ex:
        print("Exception in publishing message.")
        print(str(ex))


if __name__ == "__main__":

    file_name = "streaming_collision.csv"

    print("Loading collision records...")
    collision_rows = read_csv_file(file_name)
    print(f"Total collision records loaded: {len(collision_rows)}")

    print("Publishing records...")
    producer = connect_kafka_producer()

    current_pointer = 0
    batch_number = 1
    
    try:
        while True:
            rows_to_send = random.randint(50, 100)

            end_pointer = current_pointer + rows_to_send
            batch = collision_rows[current_pointer:end_pointer]

            print("\n" + "=" * 60)
            print(f"Batch {batch_number}")
            print("=" * 60)
            print(f"Current pointer : {current_pointer}")
            print(f"End pointer     : {end_pointer}")
            print(f"Records selected: {len(batch)}")

            # If there are no remaining records, reset the pointer for demo continuity.
            if len(batch) == 0:
                print("Reached end of file. Restarting from the beginning.")
                current_pointer = 0
                continue

            accident_ts = int(dt.datetime.now().timestamp())

            batch_to_send = []

            for row in batch:
                new_row = dict(row)
                new_row["accident_ts"] = accident_ts
                batch_to_send.append(new_row)

            publish_message(producer, topic, batch_to_send)

            # Move the pointer forward after publishing the current batch.
            current_pointer = end_pointer
            print(f"Pointer moved to: {current_pointer}")

            batch_number += 1

            if current_pointer >= len(collision_rows):
                print("Reached end of file. Restarting from the beginning.")
                current_pointer = 0

            sleep(1)

    except KeyboardInterrupt:
        print("Producer stopped by user.")

    finally:
        producer.flush()
        producer.close()
        print("Kafka producer closed.")

Loading collision records...
Total collision records loaded: 32771
Publishing records...
Kafka producer connected successfully.

Batch 1
Current pointer : 0
End pointer     : 60
Records selected: 60
Published batch successfully. Records: 60, accident_ts: 1781257944
Pointer moved to: 60

Batch 2
Current pointer : 60
End pointer     : 159
Records selected: 99
Published batch successfully. Records: 99, accident_ts: 1781257945
Pointer moved to: 159

Batch 3
Current pointer : 159
End pointer     : 214
Records selected: 55
Published batch successfully. Records: 55, accident_ts: 1781257946
Pointer moved to: 214

Batch 4
Current pointer : 214
End pointer     : 287
Records selected: 73
Published batch successfully. Records: 73, accident_ts: 1781257947
Pointer moved to: 287

Batch 5
Current pointer : 287
End pointer     : 378
Records selected: 91
Published batch successfully. Records: 91, accident_ts: 1781257948
Pointer moved to: 378

Batch 6
Current pointer : 378
End pointer     : 453
Records s


Batch 30
Current pointer : 2088
End pointer     : 2154
Records selected: 66
Published batch successfully. Records: 66, accident_ts: 1781257973
Pointer moved to: 2154

Batch 31
Current pointer : 2154
End pointer     : 2219
Records selected: 65
Published batch successfully. Records: 65, accident_ts: 1781257974
Pointer moved to: 2219

Batch 32
Current pointer : 2219
End pointer     : 2280
Records selected: 61
Published batch successfully. Records: 61, accident_ts: 1781257975
Pointer moved to: 2280

Batch 33
Current pointer : 2280
End pointer     : 2343
Records selected: 63
Published batch successfully. Records: 63, accident_ts: 1781257976
Pointer moved to: 2343

Batch 34
Current pointer : 2343
End pointer     : 2437
Records selected: 94
Published batch successfully. Records: 94, accident_ts: 1781257977
Pointer moved to: 2437

Batch 35
Current pointer : 2437
End pointer     : 2525
Records selected: 88
Published batch successfully. Records: 88, accident_ts: 1781257978
Pointer moved to: 252


Batch 59
Current pointer : 4306
End pointer     : 4363
Records selected: 57
Published batch successfully. Records: 57, accident_ts: 1781258002
Pointer moved to: 4363

Batch 60
Current pointer : 4363
End pointer     : 4418
Records selected: 55
Published batch successfully. Records: 55, accident_ts: 1781258004
Pointer moved to: 4418

Batch 61
Current pointer : 4418
End pointer     : 4498
Records selected: 80
Published batch successfully. Records: 80, accident_ts: 1781258005
Pointer moved to: 4498

Batch 62
Current pointer : 4498
End pointer     : 4574
Records selected: 76
Published batch successfully. Records: 76, accident_ts: 1781258006
Pointer moved to: 4574

Batch 63
Current pointer : 4574
End pointer     : 4667
Records selected: 93
Published batch successfully. Records: 93, accident_ts: 1781258007
Pointer moved to: 4667

Batch 64
Current pointer : 4667
End pointer     : 4723
Records selected: 56
Published batch successfully. Records: 56, accident_ts: 1781258008
Pointer moved to: 472


Batch 88
Current pointer : 6332
End pointer     : 6411
Records selected: 79
Published batch successfully. Records: 79, accident_ts: 1781258032
Pointer moved to: 6411

Batch 89
Current pointer : 6411
End pointer     : 6493
Records selected: 82
Published batch successfully. Records: 82, accident_ts: 1781258033
Pointer moved to: 6493

Batch 90
Current pointer : 6493
End pointer     : 6569
Records selected: 76
Published batch successfully. Records: 76, accident_ts: 1781258034
Pointer moved to: 6569

Batch 91
Current pointer : 6569
End pointer     : 6644
Records selected: 75
Published batch successfully. Records: 75, accident_ts: 1781258035
Pointer moved to: 6644

Batch 92
Current pointer : 6644
End pointer     : 6727
Records selected: 83
Published batch successfully. Records: 83, accident_ts: 1781258036
Pointer moved to: 6727

Batch 93
Current pointer : 6727
End pointer     : 6795
Records selected: 68
Published batch successfully. Records: 68, accident_ts: 1781258037
Pointer moved to: 679


Batch 117
Current pointer : 8541
End pointer     : 8600
Records selected: 59
Published batch successfully. Records: 59, accident_ts: 1781258061
Pointer moved to: 8600

Batch 118
Current pointer : 8600
End pointer     : 8651
Records selected: 51
Published batch successfully. Records: 51, accident_ts: 1781258062
Pointer moved to: 8651

Batch 119
Current pointer : 8651
End pointer     : 8749
Records selected: 98
Published batch successfully. Records: 98, accident_ts: 1781258063
Pointer moved to: 8749

Batch 120
Current pointer : 8749
End pointer     : 8827
Records selected: 78
Published batch successfully. Records: 78, accident_ts: 1781258064
Pointer moved to: 8827

Batch 121
Current pointer : 8827
End pointer     : 8895
Records selected: 68
Published batch successfully. Records: 68, accident_ts: 1781258065
Pointer moved to: 8895

Batch 122
Current pointer : 8895
End pointer     : 8945
Records selected: 50
Published batch successfully. Records: 50, accident_ts: 1781258066
Pointer moved t

...


Batch 1448
Current pointer : 9839
End pointer     : 9909
Records selected: 70
Published batch successfully. Records: 70, accident_ts: 1781259426
Pointer moved to: 9909

Batch 1449
Current pointer : 9909
End pointer     : 10003
Records selected: 94
Published batch successfully. Records: 94, accident_ts: 1781259427
Pointer moved to: 10003

Batch 1450
Current pointer : 10003
End pointer     : 10081
Records selected: 78
Published batch successfully. Records: 78, accident_ts: 1781259428
Pointer moved to: 10081

Batch 1451
Current pointer : 10081
End pointer     : 10173
Records selected: 92
Published batch successfully. Records: 92, accident_ts: 1781259429
Pointer moved to: 10173

Batch 1452
Current pointer : 10173
End pointer     : 10233
Records selected: 60
Published batch successfully. Records: 60, accident_ts: 1781259430
Pointer moved to: 10233

Batch 1453
Current pointer : 10233
End pointer     : 10313
Records selected: 80
Published batch successfully. Records: 80, accident_ts: 1781259


Batch 1476
Current pointer : 11981
End pointer     : 12053
Records selected: 72
Published batch successfully. Records: 72, accident_ts: 1781259454
Pointer moved to: 12053

Batch 1477
Current pointer : 12053
End pointer     : 12140
Records selected: 87
Published batch successfully. Records: 87, accident_ts: 1781259455
Pointer moved to: 12140

Batch 1478
Current pointer : 12140
End pointer     : 12217
Records selected: 77
Published batch successfully. Records: 77, accident_ts: 1781259456
Pointer moved to: 12217

Batch 1479
Current pointer : 12217
End pointer     : 12308
Records selected: 91
Published batch successfully. Records: 91, accident_ts: 1781259457
Pointer moved to: 12308

Batch 1480
Current pointer : 12308
End pointer     : 12373
Records selected: 65
Published batch successfully. Records: 65, accident_ts: 1781259458
Pointer moved to: 12373

Batch 1481
Current pointer : 12373
End pointer     : 12449
Records selected: 76
Published batch successfully. Records: 76, accident_ts: 178


Batch 1504
Current pointer : 14036
End pointer     : 14094
Records selected: 58
Published batch successfully. Records: 58, accident_ts: 1781259483
Pointer moved to: 14094

Batch 1505
Current pointer : 14094
End pointer     : 14148
Records selected: 54
Published batch successfully. Records: 54, accident_ts: 1781259484
Pointer moved to: 14148

Batch 1506
Current pointer : 14148
End pointer     : 14209
Records selected: 61
Published batch successfully. Records: 61, accident_ts: 1781259485
Pointer moved to: 14209

Batch 1507
Current pointer : 14209
End pointer     : 14297
Records selected: 88
Published batch successfully. Records: 88, accident_ts: 1781259486
Pointer moved to: 14297

Batch 1508
Current pointer : 14297
End pointer     : 14358
Records selected: 61
Published batch successfully. Records: 61, accident_ts: 1781259487
Pointer moved to: 14358

Batch 1509
Current pointer : 14358
End pointer     : 14411
Records selected: 53
Published batch successfully. Records: 53, accident_ts: 178


Batch 1532
Current pointer : 16082
End pointer     : 16171
Records selected: 89
Published batch successfully. Records: 89, accident_ts: 1781259511
Pointer moved to: 16171

Batch 1533
Current pointer : 16171
End pointer     : 16244
Records selected: 73
Published batch successfully. Records: 73, accident_ts: 1781259512
Pointer moved to: 16244

Batch 1534
Current pointer : 16244
End pointer     : 16338
Records selected: 94
Published batch successfully. Records: 94, accident_ts: 1781259513
Pointer moved to: 16338

Batch 1535
Current pointer : 16338
End pointer     : 16426
Records selected: 88
Published batch successfully. Records: 88, accident_ts: 1781259515
Pointer moved to: 16426

Batch 1536
Current pointer : 16426
End pointer     : 16518
Records selected: 92
Published batch successfully. Records: 92, accident_ts: 1781259516
Pointer moved to: 16518

Batch 1537
Current pointer : 16518
End pointer     : 16597
Records selected: 79
Published batch successfully. Records: 79, accident_ts: 178


Batch 1560
Current pointer : 18215
End pointer     : 18267
Records selected: 52
Published batch successfully. Records: 52, accident_ts: 1781259540
Pointer moved to: 18267

Batch 1561
Current pointer : 18267
End pointer     : 18335
Records selected: 68
Published batch successfully. Records: 68, accident_ts: 1781259541
Pointer moved to: 18335

Batch 1562
Current pointer : 18335
End pointer     : 18390
Records selected: 55
Published batch successfully. Records: 55, accident_ts: 1781259542
Pointer moved to: 18390

Batch 1563
Current pointer : 18390
End pointer     : 18469
Records selected: 79
Published batch successfully. Records: 79, accident_ts: 1781259543
Pointer moved to: 18469


### Producer Process Explanation

In this task, the Kafka producer is used to simulate real-time accident data streaming. The producer reads records from `streaming_collision.csv` in chronological order and sends them to the Kafka topic `a2b_accident_stream`.

The Kafka broker is accessed using the host name `kafka` and port `9092`, as defined by `bootstrap_servers=[f"{hostip}:9092"]`. In this notebook, `hostip` is set to `"kafka"`, which matches the Kafka service name in the Docker network.

A pointer-based reading process is used to control the streaming order. The variable `current_pointer` stores the current reading position in the CSV file, while `end_pointer` marks the end of the current batch. Every second, the producer randomly selects between 50 and 100 records, publishes that batch to Kafka, and then moves the pointer forward. This ensures that records are streamed sequentially, while the batch size varies to imitate a real-time data flow.

Each outgoing batch is also assigned an `accident_ts` value based on the current timestamp. This timestamp represents the simulated event time for the streamed accident records and can be used later by the Spark Structured Streaming consumer for real-time processing.

The `accident_ts` field is added by the producer to represent the simulated streaming event time. It does not replace the original accident `time` column from the dataset. The original `time` column is still used for feature engineering, while `accident_ts` is used by Spark Structured Streaming for event-time processing, watermarking, and window-based aggregation.


## Generative AI Usage Statement

Generative AI was used selectively in this assignment as a learning and support tool. The final code, testing decisions, interpretation of outputs, and notebook organisation were reviewed, adapted, and executed by me. I used Generative AI mainly to clarify concepts, improve code readability, debug errors, and draft explanatory markdown. I understand that I am responsible for the correctness, quality, and academic integrity of the submitted work.

### Task 1: Kafka Producer

For Task 1, Generative AI was used to help understand the role of the Kafka producer and how streaming records are sent to a Kafka topic. It assisted in explaining the producer logic, including how accident records are read, converted into messages, and published continuously. I used these explanations to check that my producer was correctly sending simulated streaming accident data to the expected Kafka topic. The final producer code was run and tested in my Docker environment.